# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder, Seq2Seq(Encoder + Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while문)

In [116]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import numpy as np

In [117]:
# !gdown 1qk-14tgVHPXT5jfRUE4Ua2ji4EXwS022

In [118]:
import glob
import json
import re

file_path = './qna_data/*.json'

question_inputs = []
answer_inputs = []
answer_targets = []

def preprocess_text(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))   # HTML 제거
    text = re.sub(r'[^가-힣0-9a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for file in glob.glob(file_path):
    with open(file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

        for data in json_data['data']:
            qas = data['qas'][0]
            question = preprocess_text(qas['question'])
            answer = preprocess_text(qas['answer']['text'])

            # 빈 문자열 방지
            if question and answer:
                question_inputs.append(question)
                answer_inputs.append('<sos> ' + answer)
                answer_targets.append(answer + ' <eos>')

print(len(question_inputs), len(answer_inputs), len(answer_targets))

1996 1996 1996


In [119]:
print(question_inputs[:3])
print(answer_inputs[:3])
print(answer_targets[:3])

['파스칼레 소틸레의 스파이크 높이는 몇 cm인가', '혈액 검사에서 나트륨의 정상범위는 최소 136mmol L에서 최대 몇 mmol L까지 일까', '안토닌 드보르자크의 작품 중 18개의 가곡으로 이루어진 노래는']
['<sos> 332cm', '<sos> 145', '<sos> 사이프러스']
['332cm <eos>', '145 <eos>', '사이프러스 <eos>']


In [120]:
BATCH_SIZE = 16
MAX_VOCAB_SIZE = 5000
EMBEDDING_DIM = 100
LATENT_DIM = 256
QUESTION_MAX_LEN = 30
ANSWER_MAX_LEN = 30

### 질문 토크나이징

In [121]:
question_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
question_tokenizer.fit_on_texts(question_inputs)
question_seqs = question_tokenizer.texts_to_sequences(question_inputs)

question_num_words = min(MAX_VOCAB_SIZE, len(question_tokenizer.word_index) + 1)
question_max_len = max(len(s) for s in question_seqs)

print(f'{question_num_words = }')
print(f'{question_max_len = }')

question_num_words = 5000
question_max_len = 19


### 답변 토크나이징

In [122]:
answer_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
answer_tokenizer.fit_on_texts(answer_inputs + answer_targets)

answer_input_seqs = answer_tokenizer.texts_to_sequences(answer_inputs)
answer_target_seqs = answer_tokenizer.texts_to_sequences(answer_targets)

answer_num_words = min(MAX_VOCAB_SIZE, len(answer_tokenizer.word_index) + 1)
answer_max_len = max(len(s) for s in answer_input_seqs)

print(f'{answer_num_words = }')
print(f'{answer_max_len = }')

answer_num_words = 5000
answer_max_len = 336


### 패딩 적용

In [123]:
# 패딩을 적용하여 시퀀스 길이를 맞춤
encoder_inputs = pad_sequences(question_seqs, maxlen=QUESTION_MAX_LEN, padding='pre', truncating='pre')
decoder_inputs = pad_sequences(answer_input_seqs, maxlen=ANSWER_MAX_LEN, padding='post', truncating='post')
decoder_targets = pad_sequences(answer_target_seqs, maxlen=ANSWER_MAX_LEN, padding='post', truncating='post')

print(encoder_inputs.shape)
print(decoder_inputs.shape)
print(decoder_targets.shape)

print(encoder_inputs[1000])
print([question_tokenizer.index_word[s] for s in encoder_inputs[1000] if s != 0])
print(decoder_inputs[1000])
print([answer_tokenizer.index_word[s] for s in decoder_inputs[1000] if s != 0])
print(decoder_targets[1000])
print([answer_tokenizer.index_word[s] for s in decoder_targets[1000] if s != 0])

(1996, 30)
(1996, 30)
(1996, 30)
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0 4785 1122 4786 4787  164  410 4788 4789 4790  123   97
    3 4791]
['iaaf', '이전의', '5000m', '달리기', '남자', '경기에서', 'alfred', 'shrubb가', '달성한', '경기', '결과는', '몇', '초인가']
[1 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
['<sos>', '0']
[4 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
['0', '<eos>']


In [124]:
# PyTorch Dataset 클래스 정의
# Dataset 클래스는 PyTorch에서 데이터를 로드하고 전처리하는 데 사용되는 기본 클래스입니다.
# MMTDataset 클래스는 Dataset 클래스를 상속받아 정의되며, __init__, __len__, __getitem__ 메서드를 구현합니다.

class MMTDataset(Dataset):
  def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
    super().__init__()
    self.encoder_inputs = encoder_inputs
    self.decoder_inputs = decoder_inputs
    self.decoder_targets = decoder_targets

  def __len__(self):
    return len(self.encoder_inputs)

  def __getitem__(self, index):
    return (
        torch.tensor(self.encoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_targets[index], dtype=torch.long),
    )

In [125]:
train_index, val_index = train_test_split(range(len(encoder_inputs)), random_state=0)
print(len(train_index), len(val_index))

train_dataset = MMTDataset(
    encoder_inputs[train_index],
    decoder_inputs[train_index],
    decoder_targets[train_index]
)
val_dataset = MMTDataset(
    encoder_inputs[val_index],
    decoder_inputs[val_index],
    decoder_targets[val_index]
)

1497 499


In [126]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [127]:
def make_embedding_matrix(num_words, embedding_dim=100, tokenizer=question_tokenizer):
  embedding_dict = {}

  with open('./etc/glove.6B.100d.txt', 'r', encoding='utf-8') as f:
    for line in f:
      word, *coef = line.split()
      coef = np.array(coef, dtype=float)
      embedding_dict[word] = coef

  embedding_matrix = np.zeros((num_words, embedding_dim))
  for word, index in tokenizer.word_index.items():
    if index >= num_words:
        continue
    vec = embedding_dict.get(word)
    if vec is not None:
      embedding_matrix[index] = vec
    else:
      embedding_matrix[index] = np.random.rand(embedding_dim)

  return embedding_matrix

embedding_matrix = make_embedding_matrix(question_num_words, EMBEDDING_DIM, question_tokenizer)
print(embedding_matrix.shape)

(5000, 100)


In [128]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [129]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s

In [130]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output

In [131]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(
    question_num_words,
    embedding_dim=EMBEDDING_DIM,
    latent_dim=LATENT_DIM,
    embedding_matrix=embedding_matrix
)

decoder = Decoder(
    answer_num_words,
    embedding_dim=EMBEDDING_DIM,
    latent_dim=LATENT_DIM
)

model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)
    output = output.view(-1, output.size(-1))
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/100 TrainLoss=7.7535 TrainAcc=0.0691 ValLoss=7.6014 ValAcc=0.0799
Epoch 2/100 TrainLoss=7.1224 TrainAcc=0.0797 ValLoss=7.6676 ValAcc=0.0833
Epoch 3/100 TrainLoss=6.7852 TrainAcc=0.0831 ValLoss=7.7454 ValAcc=0.0833
Epoch 4/100 TrainLoss=6.3693 TrainAcc=0.0902 ValLoss=7.8262 ValAcc=0.0857
Epoch 5/100 TrainLoss=5.8289 TrainAcc=0.1062 ValLoss=7.8725 ValAcc=0.0903
Epoch 6/100 TrainLoss=5.2289 TrainAcc=0.1434 ValLoss=7.8977 ValAcc=0.0951
Epoch 7/100 TrainLoss=4.6754 TrainAcc=0.2146 ValLoss=8.0206 ValAcc=0.0957
Epoch 8/100 TrainLoss=4.0880 TrainAcc=0.3029 ValLoss=8.1526 ValAcc=0.0969
Epoch 9/100 TrainLoss=3.5408 TrainAcc=0.3951 ValLoss=8.1888 ValAcc=0.0963
Epoch 10/100 TrainLoss=3.0700 TrainAcc=0.4818 ValLoss=8.3486 ValAcc=0.0933
Epoch 11/100 TrainLoss=2.6896 TrainAcc=0.5517 ValLoss=8.4180 ValAcc=0.0925
Epoch 12/100 TrainLoss=2.3457 TrainAcc=0.6228 ValLoss=8.5725 ValAcc=0.0867
Epoch 13/100 TrainLoss=2.0312 TrainAcc=0.6834 ValLoss=8.6596 ValAcc=0.0817
Epoch 14/100 TrainLoss=1.7955 Trai

In [132]:
torch.save(model, 'seq2seq_chatbot.pth')

In [133]:
model = torch.load('seq2seq_chatbot.pth', weights_only=False)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(5000, 100, padding_idx=0)
    (lstm): LSTM(100, 256, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(5000, 100, padding_idx=0)
    (lstm): LSTM(100, 256, batch_first=True)
    (fc): Linear(in_features=256, out_features=5000, bias=True)
  )
)

In [134]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def answer(input_seq, model, answer_tokenizer, max_len=answer_num_words, device=device):
  model = model.to(device)
  model.eval()
  encoder = model.encoder
  decoder = model.decoder

  input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

  # Encoder 처리
  with torch.no_grad():
    hidden, cell = encoder(input_seq)

  # Decoder 출력(Auto Regressive)
  sos_index = answer_tokenizer.word_index['<sos>']
  eos_index = answer_tokenizer.word_index['<eos>']

  output_sentences = []

  target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

  with torch.no_grad():
    for _ in range(max_len):
      output, hidden, cell = decoder(target_seq, hidden, cell)  # (batch_size, seq_len, vocab_size)
      proba = output.squeeze(1).softmax(dim=-1)                 # (batch_size, vocab_size)
      pred = proba.argmax(dim=-1).detach().cpu().item()

      if pred == eos_index:
        break

      if pred > 0:
        word = answer_tokenizer.index_word[pred]
        output_sentences.append(word)

      target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

  return ' '.join(output_sentences)

In [135]:
import numpy as np

for _ in range(5):
  index = np.random.choice(len(question_inputs))
  input_seq = encoder_inputs[index:index+1]
  output = answer(input_seq, model, answer_tokenizer)

  print(f'질문: {question_inputs[index]}')
  print(f'답변: {answer_inputs[index].replace("<sos> ", "")}')
  print()
  print(f'모델 추론 답변: {output}')
  print('======================================')

질문: 도미니카의 야구팀의 수는
답변: 총 6개의 야구 팀이 있다

모델 추론 답변: 총 6개의 야구 팀이 있다
질문: 엘리아 카잔의 출생지는 어디일까
답변: 터키 이스탄불

모델 추론 답변: 터키
질문: 위키미키 막내인 루시는 몇 년생일까
답변: 2002년

모델 추론 답변: 2002년
질문: 리비아는 언제 이탈리아로부터 독립했어
답변: 1951년

모델 추론 답변: 1951년
질문: 합류식 하수도의 장단점에는 어떤것들이있을까
답변: 장점 단점 관을 하나만 묻으면 되므로 비용이 저렴하고 시공이 용이하다 강우 시 수세 효과 가 있다 강우 시 우수 처리에 유리하다 침수 다발 지역 우수 배제 시설이 정비되어 있지 않은 지역에서 유리하다 관로 구경이 커서 폐쇄의 우려가 적다 검사 보수가 용이하나 청소가 오래 걸린다 사설 하수도에 연결하기 쉽다 8 유속 유량 수질의 변동폭이 크다 맑은 날씨에 수위가 낮고 유속이 낮아 고형물이 퇴적되기 쉽다 우천 시 다량의 토사가 유입된다 우천 시 계획 하수량 이상이 되면 하수의 월류 현상이 일어난다

모델 추론 답변: 시 가 있다 시 우수 유리하다 다발 지역 우수 시설이 있지 않은 지역에서 유리하다 우려가 적다 검사 오래 걸린다 쉽다 8 유량 크다 쉽다 시 시 계획 되면 현상이
